# Notebook 25 — Geodesic Transport and Residual Flow

Notebook 24 converted residual universality embeddings into continuous density, confidence, and potential fields.

Notebook 25 treats that residual universality field as a transport geometry.

Core question:

```text
How do graph families move continuously through shared residual geometry?
```

This notebook computes:

- residual kNN manifold graph,
- geodesic transport paths between topology families,
- transport-cost matrices,
- bridgeability scores,
- residual vector fields,
- bottleneck regions,
- paper-ready figures,
- CSV/JSON/Markdown summaries,
- a zip export with optional Colab download.

This version is self-contained and Colab-safe:
- it loads Notebook 24 outputs if available,
- otherwise it reconstructs a deterministic fallback residual manifold.

In [ ]:
# Setup

from pathlib import Path
import os
import json
import zipfile
import math
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

SEED = 9423
rng = np.random.default_rng(SEED)

def find_repo_root():
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    if (cwd / "notebooks").exists() or (cwd / ".git").exists() or (cwd / "results").exists():
        return cwd
    if Path("/content").exists():
        candidate = Path("/content") / "residue-manifold-learning"
        if candidate.exists():
            return candidate
        return Path("/content")
    return cwd

REPO_ROOT = find_repo_root()
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
DOCS_DIR = REPO_ROOT / "docs"
EXPORTS_DIR = REPO_ROOT / "exports"

for folder in [RESULTS_DIR, FIGURES_DIR, DOCS_DIR, EXPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("cwd:", Path.cwd())
print("repo root:", REPO_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)
print("docs:", DOCS_DIR)
print("exports:", EXPORTS_DIR)

## 1. Load residual manifold data

Preferred inputs from Notebook 24:

```text
results/residual_density_known_points.csv
results/continuous_density_grid.csv
results/continuous_density_summary.csv
```

Fallback inputs from earlier notebooks:

```text
results/residual_phase_trajectory_embedding.csv
results/residual_pca_embedding.csv
results/residual_geometry_features.csv
```

If none are available, a deterministic fallback manifold is generated so the notebook remains runnable.

In [ ]:
# Load known residual manifold points.

def read_csv_if_exists(path):
    path = Path(path)
    if path.exists():
        print("loaded:", path)
        return pd.read_csv(path), path
    return None, None

known_candidates = [
    RESULTS_DIR / "residual_density_known_points.csv",
    RESULTS_DIR / "residual_phase_trajectory_embedding.csv",
    RESULTS_DIR / "residual_pca_embedding.csv",
]

known_raw = None
known_path = None

for p in known_candidates:
    known_raw, known_path = read_csv_if_exists(p)
    if known_raw is not None:
        break

def make_fallback_known_points():
    topologies = ["ring lattice", "small world", "Erdos-Renyi", "scale free", "modular clustered"]
    Ns = [16, 32, 64, 128]

    base = {
        "ring lattice": (-2.2, 0.4),
        "small world": (-2.6, 0.9),
        "Erdos-Renyi": (-0.9, -0.1),
        "scale free": (1.5, -0.5),
        "modular clustered": (3.5, 0.1),
    }

    drift = {
        "ring lattice": [(0.4, 1.0), (0.1, 0.1), (-0.3, -0.4), (-0.5, -1.2)],
        "small world": [(-0.5, 2.6), (-0.8, 0.8), (-0.7, -1.4), (-0.6, -2.1)],
        "Erdos-Renyi": [(0.2, 1.0), (0.1, 0.3), (-0.3, -0.8), (-0.5, -1.5)],
        "scale free": [(0.6, 1.5), (-0.1, 0.2), (-0.4, -0.6), (-0.7, -0.9)],
        "modular clustered": [(0.5, 1.1), (0.2, -0.1), (-0.2, -0.7), (-0.4, -1.0)],
    }

    rows = []
    for topo in topologies:
        bx, by = base[topo]
        for N, (dx, dy) in zip(Ns, drift[topo]):
            rows.append({
                "topology": topo,
                "N": N,
                "PC1": bx + dx,
                "PC2": by + dy,
            })
    return pd.DataFrame(rows)

if known_raw is None:
    print("No known manifold file found. Using deterministic fallback scaffold.")
    known_raw = make_fallback_known_points()
    known_path = None

# Normalize columns.
known = known_raw.copy()
known.columns = [str(c).strip() for c in known.columns]

if "topology" not in known.columns:
    if "family" in known.columns:
        known["topology"] = known["family"]
    elif "label" in known.columns:
        known["topology"] = known["label"]
    else:
        known["topology"] = "unknown"

if "N" not in known.columns:
    if "n_modules" in known.columns:
        known["N"] = known["n_modules"]
    elif "graph_size" in known.columns:
        known["N"] = known["graph_size"]
    else:
        known["N"] = np.arange(len(known))

if "PC1" not in known.columns:
    if "pc1" in known.columns:
        known["PC1"] = known["pc1"]
    elif "x" in known.columns:
        known["PC1"] = known["x"]
if "PC2" not in known.columns:
    if "pc2" in known.columns:
        known["PC2"] = known["pc2"]
    elif "y" in known.columns:
        known["PC2"] = known["y"]

if "PC1" not in known.columns or "PC2" not in known.columns:
    raise ValueError("Known manifold data needs PC1/PC2 coordinate columns.")

known = known[["topology", "N", "PC1", "PC2"]].copy()
known["topology"] = known["topology"].astype(str).str.replace("_", " ")
known["topology"] = known["topology"].replace({
    "erdos renyi": "Erdos-Renyi",
    "Erdős–Rényi": "Erdos-Renyi",
})
known["N"] = pd.to_numeric(known["N"], errors="coerce").fillna(0).astype(int)
known["PC1"] = pd.to_numeric(known["PC1"], errors="coerce")
known["PC2"] = pd.to_numeric(known["PC2"], errors="coerce")
known = known.dropna(subset=["PC1", "PC2"]).sort_values(["topology", "N"]).reset_index(drop=True)

known.to_csv(RESULTS_DIR / "25_known_manifold_points.csv", index=False)

print("known path:", known_path)
print("known shape:", known.shape)
known.head()

In [ ]:
# Load density grid from Notebook 24 if available; otherwise reconstruct it.

grid_path = RESULTS_DIR / "continuous_density_grid.csv"
grid_raw, grid_loaded_path = read_csv_if_exists(grid_path)

topologies = list(known["topology"].drop_duplicates())
centroids = known.groupby("topology")[["PC1", "PC2"]].mean().loc[topologies]
points = known[["PC1", "PC2"]].to_numpy(float)

if grid_raw is not None and {"PC1", "PC2"}.issubset(grid_raw.columns):
    grid = grid_raw.copy()
    if "density" not in grid.columns:
        grid["density"] = 1.0
    if "boundary_confidence" not in grid.columns:
        centroid_dist = cdist(grid[["PC1", "PC2"]].to_numpy(float), centroids.to_numpy(float))
        sorted_dist = np.sort(centroid_dist, axis=1)
        grid["boundary_confidence"] = (sorted_dist[:, 1] - sorted_dist[:, 0]) / (sorted_dist[:, 1] + 1e-12)
    if "universality_potential" not in grid.columns:
        grid["universality_potential"] = grid["density"] * grid["boundary_confidence"]
    grid_source = "loaded continuous_density_grid.csv"
else:
    print("No continuous_density_grid.csv found. Reconstructing density grid.")
    pad = 0.75
    x_min, x_max = known["PC1"].min() - pad, known["PC1"].max() + pad
    y_min, y_max = known["PC2"].min() - pad, known["PC2"].max() + pad
    xs = np.linspace(x_min, x_max, 220)
    ys = np.linspace(y_min, y_max, 220)
    XX, YY = np.meshgrid(xs, ys)
    grid_xy = np.column_stack([XX.ravel(), YY.ravel()])

    D = pairwise_distances(points)
    nonzero = D[D > 0]
    bandwidth = float(np.median(nonzero) * 0.45) if len(nonzero) else 1.0
    bandwidth = max(bandwidth, 0.35)

    dist2 = cdist(grid_xy, points) ** 2
    density = np.exp(-0.5 * dist2 / (bandwidth ** 2)).sum(axis=1)
    density = density / (density.max() + 1e-12)

    centroid_dist = cdist(grid_xy, centroids.to_numpy(float))
    sorted_dist = np.sort(centroid_dist, axis=1)
    nearest_idx = np.argmin(centroid_dist, axis=1)

    confidence = (sorted_dist[:, 1] - sorted_dist[:, 0]) / (sorted_dist[:, 1] + 1e-12)
    confidence = np.clip(confidence, 0, 1)

    grid = pd.DataFrame({
        "PC1": grid_xy[:, 0],
        "PC2": grid_xy[:, 1],
        "density": density,
        "nearest_family": [topologies[i] for i in nearest_idx],
        "boundary_confidence": confidence,
        "universality_potential": density * confidence,
    })
    grid.to_csv(RESULTS_DIR / "continuous_density_grid.csv", index=False)
    grid_source = "reconstructed density grid"

grid = grid.replace([np.inf, -np.inf], np.nan).dropna(subset=["PC1", "PC2"])
for col in ["density", "boundary_confidence", "universality_potential"]:
    grid[col] = pd.to_numeric(grid[col], errors="coerce").fillna(0)

print("grid source:", grid_source)
print("grid shape:", grid.shape)
grid.head()

## 2. Construct residual kNN manifold graph

Graph nodes are sampled from the continuous density grid plus known family points.

Edge weights penalize:

- Euclidean residual distance,
- low density,
- low boundary confidence.

The resulting graph defines a transport geometry over the residual manifold.

In [ ]:
# Sample graph nodes from grid + known points.

MAX_GRID_NODES = 1800
grid_sample = grid.copy()

# Prefer nodes with meaningful density/potential.
if len(grid_sample) > MAX_GRID_NODES:
    weights = grid_sample["density"].to_numpy(float) + 0.15
    weights = weights / weights.sum()
    idx = rng.choice(len(grid_sample), size=MAX_GRID_NODES, replace=False, p=weights)
    grid_sample = grid_sample.iloc[idx].copy()

grid_nodes = grid_sample[["PC1", "PC2", "density", "boundary_confidence", "universality_potential"]].copy()
grid_nodes["node_type"] = "grid"
grid_nodes["topology"] = "grid"
grid_nodes["N"] = -1

known_nodes = known.copy()
known_nodes["density"] = 1.0
known_nodes["boundary_confidence"] = 1.0
known_nodes["universality_potential"] = 1.0
known_nodes["node_type"] = "known"

nodes = pd.concat([grid_nodes, known_nodes], ignore_index=True)
nodes = nodes.reset_index(drop=True)
nodes["node_id"] = np.arange(len(nodes))

coords = nodes[["PC1", "PC2"]].to_numpy(float)

print("nodes:", nodes.shape)
nodes.head()

In [ ]:
# Build kNN graph with transport weights.

K = 10
nbrs = NearestNeighbors(n_neighbors=min(K + 1, len(nodes)), algorithm="auto")
nbrs.fit(coords)
distances, indices = nbrs.kneighbors(coords)

edge_rows = []
row_idx = []
col_idx = []
weights = []

for i in range(len(nodes)):
    for d, j in zip(distances[i, 1:], indices[i, 1:]):
        if i == j:
            continue

        # average local density/confidence along edge
        dens = 0.5 * (nodes.loc[i, "density"] + nodes.loc[j, "density"])
        conf = 0.5 * (nodes.loc[i, "boundary_confidence"] + nodes.loc[j, "boundary_confidence"])
        pot = 0.5 * (nodes.loc[i, "universality_potential"] + nodes.loc[j, "universality_potential"])

        # cost increases through sparse or low-confidence regions
        density_penalty = 1.0 / (0.15 + dens)
        confidence_penalty = 1.0 / (0.15 + conf)
        potential_penalty = 1.0 / (0.20 + pot)

        w = float(d * (0.45 * density_penalty + 0.35 * confidence_penalty + 0.20 * potential_penalty))

        row_idx.append(i)
        col_idx.append(j)
        weights.append(w)

        edge_rows.append({
            "source": i,
            "target": int(j),
            "euclidean_distance": float(d),
            "density_mean": float(dens),
            "confidence_mean": float(conf),
            "potential_mean": float(pot),
            "transport_weight": w,
        })

edges = pd.DataFrame(edge_rows)
edges.to_csv(RESULTS_DIR / "25_weighted_edges.csv", index=False)

adj = csr_matrix((weights, (row_idx, col_idx)), shape=(len(nodes), len(nodes)))
# symmetrize
adj = adj.minimum(adj.T) + (adj.maximum(adj.T) - adj.minimum(adj.T))

print("edges:", edges.shape)
edges.head()

## 3. Residual kNN graph visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Plot a subset of edges for readability.
edge_plot = edges.sample(min(len(edges), 2500), random_state=SEED) if len(edges) > 2500 else edges

for _, e in edge_plot.iterrows():
    a = nodes.loc[int(e["source"])]
    b = nodes.loc[int(e["target"])]
    ax.plot([a["PC1"], b["PC1"]], [a["PC2"], b["PC2"]], color="gray", alpha=0.08, linewidth=0.6)

sc = ax.scatter(
    nodes.loc[nodes["node_type"] == "grid", "PC1"],
    nodes.loc[nodes["node_type"] == "grid", "PC2"],
    c=nodes.loc[nodes["node_type"] == "grid", "universality_potential"],
    s=8,
    alpha=0.5,
)
plt.colorbar(sc, ax=ax, label="universality potential")

for topo in topologies:
    sub = known[known["topology"] == topo]
    ax.plot(sub["PC1"], sub["PC2"], marker="o", linewidth=2.2, label=topo)

ax.set_title("Residual kNN transport graph")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(loc="best")
ax.grid(alpha=0.3)

fig.tight_layout()
fig_path = FIGURES_DIR / "25_residual_knn_graph.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", fig_path)

## 4. Geodesic transport paths between topology families

We compute graph-geodesic distances between family centroids and extract shortest paths.

In [ ]:
# Find nearest graph node to each family centroid.

centroid_nodes = {}
for topo in topologies:
    c = centroids.loc[topo].to_numpy(float).reshape(1, -1)
    d = cdist(c, coords).ravel()
    centroid_nodes[topo] = int(np.argmin(d))

centroid_nodes

In [ ]:
# All-pairs Dijkstra from family centroid nodes.
source_nodes = [centroid_nodes[t] for t in topologies]
dist_matrix, predecessors = dijkstra(
    csgraph=adj,
    directed=False,
    indices=source_nodes,
    return_predecessors=True
)

geo = pd.DataFrame(index=topologies, columns=topologies, dtype=float)
for i, t1 in enumerate(topologies):
    for j, t2 in enumerate(topologies):
        target = centroid_nodes[t2]
        geo.loc[t1, t2] = dist_matrix[i, target]

geo.to_csv(RESULTS_DIR / "25_geodesic_distance_matrix.csv")
geo

In [ ]:
def reconstruct_path_from_predecessors(source_row_idx, target_node):
    source_node = source_nodes[source_row_idx]
    path = [target_node]
    current = target_node

    max_steps = len(nodes) + 5
    steps = 0

    while current != source_node and current != -9999 and steps < max_steps:
        current = int(predecessors[source_row_idx, current])
        if current == -9999:
            return []
        path.append(current)
        steps += 1

    path.reverse()
    return path

bridge_paths = {}

for i, t1 in enumerate(topologies):
    for t2 in topologies:
        if t1 == t2:
            continue
        target = centroid_nodes[t2]
        path = reconstruct_path_from_predecessors(i, target)
        bridge_paths[f"{t1} -> {t2}"] = path

bridge_summary_rows = []

for key, path in bridge_paths.items():
    if not path:
        continue
    sub = nodes.loc[path]
    bridge_summary_rows.append({
        "bridge": key,
        "n_path_nodes": len(path),
        "mean_density": float(sub["density"].mean()),
        "mean_confidence": float(sub["boundary_confidence"].mean()),
        "mean_potential": float(sub["universality_potential"].mean()),
        "min_potential": float(sub["universality_potential"].min()),
    })

bridge_summary = pd.DataFrame(bridge_summary_rows)
bridge_summary.to_csv(RESULTS_DIR / "25_family_bridge_paths_summary.csv", index=False)

json_paths = {k: [int(x) for x in v] for k, v in bridge_paths.items()}
(RESULTS_DIR / "25_family_bridge_paths.json").write_text(json.dumps(json_paths, indent=2))

bridge_summary.head()

## 5. Geodesic transport overlay

In [ ]:
# Select readable set of bridges.
selected_bridges = []
preferred = [
    "ring lattice -> small world",
    "small world -> Erdos-Renyi",
    "Erdos-Renyi -> scale free",
    "scale free -> modular clustered",
    "ring lattice -> modular clustered",
]
for b in preferred:
    if b in bridge_paths and bridge_paths[b]:
        selected_bridges.append(b)

if not selected_bridges:
    selected_bridges = [k for k, v in bridge_paths.items() if v][:5]

fig, ax = plt.subplots(figsize=(10, 7))

sc = ax.scatter(grid["PC1"], grid["PC2"], c=grid["universality_potential"], s=4, alpha=0.25)
plt.colorbar(sc, ax=ax, label="universality potential")

for topo in topologies:
    sub = known[known["topology"] == topo]
    ax.plot(sub["PC1"], sub["PC2"], marker="o", linewidth=1.8, alpha=0.7, label=topo)
    c = centroids.loc[topo]
    ax.scatter(c["PC1"], c["PC2"], marker="*", s=220, color="black")
    ax.text(c["PC1"], c["PC2"], topo, fontsize=9, weight="bold")

for b in selected_bridges:
    path_nodes = bridge_paths[b]
    sub = nodes.loc[path_nodes]
    ax.plot(sub["PC1"], sub["PC2"], linewidth=3, alpha=0.85, label=f"path: {b}")

ax.set_title("Geodesic transport paths over residual manifold")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(loc="best", fontsize=8)
ax.grid(alpha=0.3)

fig.tight_layout()
fig_path = FIGURES_DIR / "25_geodesic_transport_paths.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", fig_path)

## 6. Transport cost and bridgeability matrices

In [ ]:
# Transport cost matrix = geodesic distances.
transport_cost = geo.copy()
transport_cost.to_csv(RESULTS_DIR / "25_transport_cost_matrix.csv")

# Bridgeability score combines path density/confidence/potential with inverse geodesic cost.
bridge_rows = []
for key, path in bridge_paths.items():
    if not path:
        continue
    source, target = key.split(" -> ")
    sub = nodes.loc[path]
    cost = float(transport_cost.loc[source, target])
    mean_density = float(sub["density"].mean())
    mean_conf = float(sub["boundary_confidence"].mean())
    mean_potential = float(sub["universality_potential"].mean())
    score = (mean_density * mean_conf + 0.25 * mean_potential) / (cost + 1e-9)

    bridge_rows.append({
        "source_family": source,
        "target_family": target,
        "geodesic_cost": cost,
        "mean_density": mean_density,
        "mean_confidence": mean_conf,
        "mean_potential": mean_potential,
        "bridgeability_score": score,
    })

bridge_scores = pd.DataFrame(bridge_rows)
bridge_scores.to_csv(RESULTS_DIR / "25_manifold_bridge_scores.csv", index=False)

bridge_matrix = pd.DataFrame(index=topologies, columns=topologies, dtype=float)
for _, row in bridge_scores.iterrows():
    bridge_matrix.loc[row["source_family"], row["target_family"]] = row["bridgeability_score"]
for topo in topologies:
    bridge_matrix.loc[topo, topo] = np.nan

bridge_matrix.to_csv(RESULTS_DIR / "25_bridgeability_matrix.csv")

bridge_scores.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

im0 = axes[0].imshow(transport_cost.to_numpy(float), aspect="auto")
axes[0].set_title("Geodesic transport cost")
axes[0].set_xticks(range(len(topologies)))
axes[0].set_yticks(range(len(topologies)))
axes[0].set_xticklabels(topologies, rotation=45, ha="right")
axes[0].set_yticklabels(topologies)
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

B = bridge_matrix.to_numpy(float)
im1 = axes[1].imshow(B, aspect="auto")
axes[1].set_title("Universality bridgeability score")
axes[1].set_xticks(range(len(topologies)))
axes[1].set_yticks(range(len(topologies)))
axes[1].set_xticklabels(topologies, rotation=45, ha="right")
axes[1].set_yticklabels(topologies)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.tight_layout()
fig_path = FIGURES_DIR / "25_bridge_matrix_heatmap.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", fig_path)

## 7. Residual transport vector field

We estimate a simple flow field from gradients of density and boundary confidence.

In [ ]:
# Build regular grid from existing density grid if it is rectangular enough.
# Otherwise interpolate by nearest neighbors onto a new regular grid.

x_min, x_max = grid["PC1"].min(), grid["PC1"].max()
y_min, y_max = grid["PC2"].min(), grid["PC2"].max()

NX, NY = 42, 42
gx = np.linspace(x_min, x_max, NX)
gy = np.linspace(y_min, y_max, NY)
GXX, GYY = np.meshgrid(gx, gy)
sample_xy = np.column_stack([GXX.ravel(), GYY.ravel()])

grid_xy = grid[["PC1", "PC2"]].to_numpy(float)
nearest = np.argmin(cdist(sample_xy, grid_xy), axis=1)

density_field = grid.iloc[nearest]["density"].to_numpy(float).reshape(NY, NX)
confidence_field = grid.iloc[nearest]["boundary_confidence"].to_numpy(float).reshape(NY, NX)
potential_field = grid.iloc[nearest]["universality_potential"].to_numpy(float).reshape(NY, NX)

# Gradients.
dy, dx = np.gradient(potential_field)
flow_x = dx
flow_y = dy

norm = np.sqrt(flow_x**2 + flow_y**2) + 1e-9
flow_xn = flow_x / norm
flow_yn = flow_y / norm

flow_samples = pd.DataFrame({
    "PC1": GXX.ravel(),
    "PC2": GYY.ravel(),
    "density": density_field.ravel(),
    "confidence": confidence_field.ravel(),
    "potential": potential_field.ravel(),
    "flow_x": flow_x.ravel(),
    "flow_y": flow_y.ravel(),
    "flow_x_unit": flow_xn.ravel(),
    "flow_y_unit": flow_yn.ravel(),
})
flow_samples.to_csv(RESULTS_DIR / "25_flow_field_samples.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.contourf(GXX, GYY, potential_field, levels=24, alpha=0.75)
plt.colorbar(im, ax=ax, label="universality potential")

skip = 3
ax.quiver(
    GXX[::skip, ::skip],
    GYY[::skip, ::skip],
    flow_xn[::skip, ::skip],
    flow_yn[::skip, ::skip],
    color="black",
    alpha=0.65,
    scale=35,
)

for topo in topologies:
    sub = known[known["topology"] == topo]
    ax.plot(sub["PC1"], sub["PC2"], marker="o", linewidth=1.8, label=topo)

ax.set_title("Residual transport vector field")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()

fig_path = FIGURES_DIR / "25_transport_vector_field.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", fig_path)

## 8. Curvature-weighted flow and bottleneck regions

In [ ]:
# Trajectory curvature from known topology paths.

curvature_rows = []

for topo in topologies:
    sub = known[known["topology"] == topo].sort_values("N")
    pts = sub[["PC1", "PC2"]].to_numpy(float)

    if len(pts) < 3:
        continue

    local_curv = np.zeros(len(pts))
    for i in range(1, len(pts) - 1):
        kappa_vec = pts[i + 1] - 2 * pts[i] + pts[i - 1]
        local_curv[i] = np.linalg.norm(kappa_vec)

    for row_i, (_, row) in enumerate(sub.iterrows()):
        curvature_rows.append({
            "topology": topo,
            "N": int(row["N"]),
            "PC1": row["PC1"],
            "PC2": row["PC2"],
            "local_curvature": float(local_curv[row_i]),
        })

curvature_df = pd.DataFrame(curvature_rows)
curvature_df.to_csv(RESULTS_DIR / "25_trajectory_curvature.csv", index=False)

curvature_df.head()

In [ ]:
# Bottlenecks: low density, low confidence, low potential.
bottleneck_score = (
    (1 - grid["density"].to_numpy(float))
    * (1 - grid["boundary_confidence"].to_numpy(float))
    * (1 - grid["universality_potential"].to_numpy(float))
)

grid_bottlenecks = grid.copy()
grid_bottlenecks["bottleneck_score"] = bottleneck_score

threshold = np.quantile(bottleneck_score, 0.97)
bottlenecks = grid_bottlenecks[grid_bottlenecks["bottleneck_score"] >= threshold].copy()
bottlenecks.to_csv(RESULTS_DIR / "25_bottleneck_regions.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(grid["PC1"], grid["PC2"], c=bottleneck_score, s=8, alpha=0.55)
plt.colorbar(sc, ax=ax, label="bottleneck score")

ax.scatter(bottlenecks["PC1"], bottlenecks["PC2"], s=18, c="red", alpha=0.75, label="top bottlenecks")

for topo in topologies:
    sub = known[known["topology"] == topo].sort_values("N")
    ax.plot(sub["PC1"], sub["PC2"], marker="o", linewidth=1.8, label=topo)

ax.set_title("Residual bottleneck regions")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(loc="best", fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()

fig_path = FIGURES_DIR / "25_bottleneck_regions.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", fig_path)
print("bottleneck rows:", bottlenecks.shape)

## 9. Transport phase diagram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(
    bridge_scores["geodesic_cost"],
    bridge_scores["bridgeability_score"],
    s=90,
    alpha=0.75,
)
for _, row in bridge_scores.iterrows():
    axes[0].text(
        row["geodesic_cost"],
        row["bridgeability_score"],
        f"{row['source_family']}→{row['target_family']}",
        fontsize=7,
        alpha=0.8,
    )
axes[0].set_xlabel("geodesic cost")
axes[0].set_ylabel("bridgeability score")
axes[0].set_title("Bridgeability vs transport cost")
axes[0].grid(alpha=0.3)

axes[1].scatter(
    bridge_scores["mean_confidence"],
    bridge_scores["mean_density"],
    c=bridge_scores["geodesic_cost"],
    s=90,
    alpha=0.75,
)
axes[1].set_xlabel("mean confidence along path")
axes[1].set_ylabel("mean density along path")
axes[1].set_title("Transport path density-confidence phase diagram")
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig_path = FIGURES_DIR / "25_transport_cost_phase_diagram.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", fig_path)

## 10. Export interpretation summary

In [ ]:
summary = {
    "notebook": "25_geodesic_transport_and_residual_flow.ipynb",
    "created_at_utc": datetime.utcnow().isoformat(),
    "source_file": str(known_path) if known_path else "deterministic_fallback_scaffold",
    "grid_source": grid_source,
    "n_known_points": int(len(known)),
    "n_grid_points": int(len(grid)),
    "n_graph_nodes": int(len(nodes)),
    "n_edges": int(len(edges)),
    "topologies": topologies,
    "figures": sorted([p.name for p in FIGURES_DIR.glob("25_*.png")]),
    "results": sorted([p.name for p in RESULTS_DIR.glob("25_*")]),
    "interpretation": {
        "core_question": "How do graph families move continuously through shared residual geometry?",
        "core_claim": (
            "Residual universality can be represented as a transport geometry where "
            "families connect through density-supported, confidence-weighted geodesic paths."
        ),
    },
}

summary_path = RESULTS_DIR / "25_geodesic_transport_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

md_text = f'''# Notebook 25 — Geodesic Transport and Residual Flow

Generated: {summary["created_at_utc"]} UTC

## Core question

How do graph families move continuously through shared residual geometry?

## Core claim

Residual universality can be represented as a transport geometry where families connect through density-supported, confidence-weighted geodesic paths.

## Main outputs

### Figures
{chr(10).join("- `figures/" + name + "`" for name in summary["figures"])}

### Results
{chr(10).join("- `results/" + name + "`" for name in summary["results"])}

## Source

`{summary["source_file"]}`
'''

doc_path = DOCS_DIR / "notebook_25_geodesic_transport_summary.md"
doc_path.write_text(md_text)

print(json.dumps(summary, indent=2))
print("saved:", summary_path)
print("saved:", doc_path)

## 11. Export zip / optional Colab download

This cell is consistent with the earlier notebooks:

- creates a manifest,
- zips Notebook 25 figures/results/docs/manifest,
- includes the notebook file if it exists in `notebooks/`,
- optionally triggers a Colab download.

In [ ]:
# Export Notebook 25 outputs + optional Colab download

manifest = {
    "notebook": "25_geodesic_transport_and_residual_flow.ipynb",
    "created_outputs": {
        "figures": sorted([p.name for p in FIGURES_DIR.glob("25_*.png")]),
        "results": sorted([p.name for p in RESULTS_DIR.glob("25_*")]),
        "docs": sorted([p.name for p in DOCS_DIR.glob("notebook_25*")]),
    },
    "source_file": str(known_path) if known_path else "deterministic_fallback_scaffold",
}

manifest_path = EXPORTS_DIR / "25_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

zip_path = EXPORTS_DIR / "25_geodesic_transport_and_residual_flow_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    # notebook itself, if running from repo
    notebook_candidates = [
        REPO_ROOT / "notebooks" / "25_geodesic_transport_and_residual_flow.ipynb",
        REPO_ROOT / "25_geodesic_transport_and_residual_flow.ipynb",
    ]

    for notebook_path in notebook_candidates:
        if notebook_path.exists():
            z.write(notebook_path, arcname="notebooks/25_geodesic_transport_and_residual_flow.ipynb")
            break

    for p in FIGURES_DIR.glob("25_*.png"):
        z.write(p, arcname=f"figures/{p.name}")

    for p in RESULTS_DIR.glob("25_*"):
        z.write(p, arcname=f"results/{p.name}")

    for p in DOCS_DIR.glob("notebook_25*"):
        z.write(p, arcname=f"docs/{p.name}")

    z.write(manifest_path, arcname="exports/25_manifest.json")

print("Wrote:", zip_path)
print("Zip size MB:", round(zip_path.stat().st_size / 1e6, 3))

# Optional Colab download:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Colab download skipped. Download manually from:", zip_path)